In [5]:
import requests
import subprocess
import json
import csv
import os
import io
import pandas as pd
import xml.etree.ElementTree as ET
import zipfile
import re
from typing import List, Dict, Optional, Tuple
from openpyxl import load_workbook

# Establece los parámetros necesarios para la solicitud
client_id = '58191b89-cee4-11ed-a09d-ee50c5eb4bb5'
scope = 'b2bapiclub-api'
grant_type = 'password'
username = 'b2bvillarealcf@mediacoach.es'
password = 'r728-FHj3RE!'

# URL a la que se hará la solicitud de token
token_url = 'https://id.mediacoach.es/connect/token'

# Datos que se enviarán en la solicitud
data = {
    'client_id': client_id,
    'scope': scope,
    'grant_type': grant_type,
    'username': username,
    'password': password
}

# Encabezados para la solicitud
headers = {
    'Content-Type': 'application/x-www-form-urlencoded'
}

# Realiza la solicitud POST
response = requests.post(token_url, data=data, headers=headers)

# Verifica si la solicitud fue exitosa
if response.status_code == 200:
    # Convierte la respuesta en JSON y obtiene el access_token
    token_response = response.json()
    AccessToken = token_response.get('access_token', '')
    expires_in = token_response.get('expires_in', '')
    token_type = token_response.get('token_type', '')
    scope = token_response.get('scope', '')

    # Imprime el access_token y otros detalles
    print(f'Access Token: {AccessToken}')
    print(f'Expires In: {expires_in}')
    print(f'Token Type: {token_type}')
    print(f'Scope: {scope}')
else:
    # Imprime el mensaje de error si la solicitud no fue exitosa
    print(f'Error: {response.status_code}')
    print(response.text)

# Subscripción del villarreal
SubscriptionKey = '729f9154234d4ff3bb0a692c6a0510c4'
# Url base de la api de media coach
api_url_base = "https://club-api.mediacoach.es"

# Función para hacer consultar curl
credenciales = f"--header 'Ocp-Apim-Subscription-Key: {SubscriptionKey}' --header 'Authorization: Bearer {AccessToken}'"

def ejecutar_curl_comando(comando):
    process = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print("Error en el comando curl:")
        print(stderr.decode())
        return None
    return json.loads(stdout)

Access Token: 0A446BA2063DC3C8621575A99DA5DB656F6316D6E2C135FD7223E5AC676E5DEF
Expires In: 10800
Token Type: Bearer
Scope: b2bapiclub-api


In [6]:
# =====================================================================================
# FUNCIONES MEJORADAS CON PROCESADORES ROBUSTOS PARA MEDIACOACH
# =====================================================================================

import zipfile
import re
from typing import List, Dict, Optional, Tuple

# =====================================================================================
# PROCESADOR ROBUSTO PARA ARCHIVOS XLSX (RENDIMIENTO Y MÁXIMA EXIGENCIA)
# =====================================================================================

def xlsx_to_csv_robusto(xlsx_content, csv_path):
    """Convierte contenido XLSX a CSV usando múltiples estrategias robustas"""
    print(f"🔍 Procesando XLSX: {len(xlsx_content)} bytes")
    
    # Validaciones previas
    if not xlsx_content or len(xlsx_content) < 100:
        print(f"❌ XLSX: Archivo muy pequeño o vacío")
        return False
    
    # Verificar que sea realmente un archivo Excel válido
    if not (xlsx_content.startswith(b'PK') or xlsx_content.startswith(b'\\xd0\\xcf')):
        print(f"❌ XLSX: No es un archivo Excel válido")
        return False
    
    # Intentar diferentes engines y configuraciones
    estrategias_lectura = [
        ('openpyxl', {'data_only': True, 'read_only': True}),
        ('openpyxl', {'data_only': True}),
        ('openpyxl', {'read_only': True}),
        ('openpyxl', {}),
    ]
    
    # Intentar con calamine si está disponible
    try:
        import calamine
        estrategias_lectura.append(('calamine', {}))
    except ImportError:
        pass
    
    df_raw = None
    estrategia_exitosa = None
    
    for engine, kwargs in estrategias_lectura:
        try:
            print(f"   🔧 Probando {engine} con {kwargs}")
            
            # Combinar argumentos
            read_args = {'header': None, 'engine': engine}
            read_args.update(kwargs)
            
            df_raw = pd.read_excel(io.BytesIO(xlsx_content), **read_args)
            estrategia_exitosa = f"{engine} {kwargs}"
            print(f"   ✅ Estrategia exitosa: {estrategia_exitosa} - Dimensiones: {df_raw.shape}")
            break
            
        except ImportError as e:
            print(f"   ⚠️ Engine {engine} no disponible")
            continue
        except Exception as e:
            print(f"   ❌ {engine} falló: {str(e)[:100]}")
            continue
    
    # Si todos fallan, intentar método de rescate con zipfile
    if df_raw is None or df_raw.empty:
        print(f"   🔧 Intentando método de rescate con zipfile...")
        df_raw = extraer_xlsx_con_zipfile(xlsx_content)
        if df_raw is not None:
            estrategia_exitosa = "zipfile rescue"
            print(f"   ✅ Rescate exitoso - Dimensiones: {df_raw.shape}")
    
    if df_raw is None or df_raw.empty:
        print(f"❌ XLSX: IMPOSIBLE LEER ARCHIVO")
        return False
    
    try:
        # ESTRUCTURA ESPECÍFICA DE MEDIACOACH:
        # - Info del partido en COLUMNA E, FILA 2 (índice 1)
        # - Headers en FILA 6 (índice 5)
        # - Datos desde FILA 7 en adelante
        
        # Extraer info del partido
        partido_info = extraer_info_partido_columna_e(df_raw)
        print(f"   📊 Info partido: {list(partido_info.keys())}")
        
        # Headers en fila 6 (confirmado por la estructura)
        fila_headers = 5
        
        if len(df_raw) <= fila_headers:
            print(f"❌ XLSX: No tiene suficientes filas (necesita > {fila_headers})")
            return False
        
        # Extraer headers
        headers = limpiar_headers(df_raw.iloc[fila_headers].tolist())
        print(f"   🏷️  Headers: {headers[:3]}... ({len(headers)} total)")
        
        # Extraer datos desde fila 7
        df_datos = df_raw.iloc[fila_headers + 1:].copy()
        
        # Ajustar columnas
        if len(headers) != len(df_datos.columns):
            min_cols = min(len(headers), len(df_datos.columns))
            headers = headers[:min_cols]
            df_datos = df_datos.iloc[:, :min_cols]
        
        df_datos.columns = headers
        df_datos = df_datos.dropna(how='all')
        
        if df_datos.empty:
            print(f"❌ XLSX: Datos vacíos después de limpieza")
            return False
        
        # Añadir info del partido como columnas
        for key, value in partido_info.items():
            df_datos[f'partido_{key}'] = value
        
        # Guardar como CSV
        df_datos.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"✅ XLSX → CSV: {len(df_datos)} filas, {len(df_datos.columns)} columnas")
        return True
        
    except Exception as e:
        print(f"❌ Error procesando estructura XLSX: {e}")
        return False

def extraer_xlsx_con_zipfile(content: bytes) -> Optional[pd.DataFrame]:
    """Método de rescate: extraer datos XLSX usando zipfile directamente"""
    try:
        with zipfile.ZipFile(io.BytesIO(content), 'r') as zip_ref:
            # Buscar worksheets
            worksheet_files = [f for f in zip_ref.namelist() if f.startswith('xl/worksheets/')]
            if not worksheet_files:
                return None
            
            # Usar la primera worksheet
            worksheet_file = worksheet_files[0]
            
            # Leer el XML
            with zip_ref.open(worksheet_file) as ws_file:
                ws_content = ws_file.read()
            
            # Parsear XML básico
            root = ET.fromstring(ws_content)
            
            # Namespace de Excel
            ns = {'': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            
            rows_data = []
            for row_elem in root.findall('.//row', ns):
                row_data = []
                for cell_elem in row_elem.findall('.//c', ns):
                    value_elem = cell_elem.find('v', ns)
                    if value_elem is not None:
                        row_data.append(value_elem.text)
                    else:
                        row_data.append('')
                
                if row_data:
                    rows_data.append(row_data)
            
            if not rows_data:
                return None
            
            # Crear DataFrame
            max_cols = max(len(row) for row in rows_data) if rows_data else 0
            for row in rows_data:
                while len(row) < max_cols:
                    row.append('')
            
            return pd.DataFrame(rows_data)
            
    except Exception as e:
        print(f"Error en rescate zipfile: {e}")
        return None

def extraer_info_partido_columna_e(df: pd.DataFrame) -> Dict:
    """Extrae información del partido desde COLUMNA E, FILA 2"""
    info = {}
    
    try:
        # Verificar que existe columna E (índice 4) y fila 2 (índice 1)
        if len(df) > 1 and len(df.columns) > 4:
            valor_partido = df.iloc[1, 4]  # Fila 2, Columna E
            
            if pd.notna(valor_partido) and isinstance(valor_partido, str):
                texto = valor_partido.strip()
                
                # FORMATO: "LALIGA EA SPORTS | Temporada 2024 - 2025 | J3 | Real Betis 2 - 1 Getafe CF (2024-09-18)"
                if '|' in texto:
                    partes = [parte.strip() for parte in texto.split('|')]
                    
                    if len(partes) >= 4:
                        info['competicion'] = partes[0]
                        
                        # Temporada
                        if 'Temporada' in partes[1]:
                            info['temporada_info'] = partes[1]
                            temporada_match = re.search(r'(\\d{4})\\s*-\\s*(\\d{4})', partes[1])
                            if temporada_match:
                                info['temporada_inicio'] = temporada_match.group(1)
                                info['temporada_fin'] = temporada_match.group(2)
                        
                        # Jornada
                        jornada_match = re.search(r'(J\\d+)', partes[2])
                        if jornada_match:
                            info['jornada'] = jornada_match.group(1)
                        
                        # Partido y fecha
                        if len(partes) > 3:
                            partido_parte = partes[-1]
                            
                            # Extraer fecha
                            fecha_match = re.search(r'\\((\\d{4}-\\d{2}-\\d{2})\\)', partido_parte)
                            if fecha_match:
                                info['fecha'] = fecha_match.group(1)
                                partido_sin_fecha = re.sub(r'\\s*\\(\\d{4}-\\d{2}-\\d{2}\\)', '', partido_parte).strip()
                            else:
                                partido_sin_fecha = partido_parte.strip()
                            
                            # Extraer equipos y resultado
                            resultado_match = re.search(r'^(.+?)\\s+(\\d+)\\s*-\\s*(\\d+)\\s+(.+)$', partido_sin_fecha)
                            if resultado_match:
                                info['equipo_local'] = resultado_match.group(1).strip()
                                info['goles_local'] = int(resultado_match.group(2))
                                info['goles_visitante'] = int(resultado_match.group(3))
                                info['equipo_visitante'] = resultado_match.group(4).strip()
                
                elif texto:
                    info['partido_raw'] = texto
                    
                    # Buscar fecha
                    fecha_match = re.search(r'(\\d{4}-\\d{2}-\\d{2})', texto)
                    if fecha_match:
                        info['fecha'] = fecha_match.group(1)
    
    except Exception as e:
        print(f"Error extrayendo info de partido: {e}")
    
    return info

def limpiar_headers(headers: List) -> List[str]:
    """Limpia los nombres de headers"""
    headers_limpios = []
    for i, header in enumerate(headers):
        if pd.notna(header) and str(header).strip():
            nombre = str(header).strip()
            nombre = re.sub(r'[^\\w\\s\\(\\)\\%\\-]', '_', nombre)
            nombre = re.sub(r'\\s+', '_', nombre)
            nombre = re.sub(r'_+', '_', nombre).strip('_')
            headers_limpios.append(nombre or f'columna_{i+1}')
        else:
            headers_limpios.append(f'columna_{i+1}')
    return headers_limpios

# =====================================================================================
# PROCESADOR ROBUSTO PARA ARCHIVOS XML (BEYOND STATS Y MÁXIMA EXIGENCIA REVISADA)
# =====================================================================================

def xml_to_csv_robusto(xml_content, csv_path):
    """Convierte contenido XML a CSV usando estructura específica de MediaCoach"""
    try:
        root = ET.fromstring(xml_content)
        instances = root.findall('.//instance')
        
        if not instances:
            print(f"❌ XML: No se encontraron instances")
            return False
        
        datos = []
        
        for instance in instances:
            registro = procesar_instancia_xml(instance)
            if registro:
                datos.append(registro)
        
        if not datos:
            print(f"❌ XML: No se pudieron extraer datos")
            return False
        
        # Crear DataFrame y guardar
        df = pd.DataFrame(datos)
        df.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"✅ XML → CSV: {len(df)} registros, {len(df.columns)} columnas")
        return True
        
    except Exception as e:
        print(f"❌ Error procesando XML: {e}")
        return False

def procesar_instancia_xml(instance) -> Optional[Dict]:
    """Procesa una instancia XML de MediaCoach"""
    try:
        # Elementos básicos
        id_elem = instance.find('ID')
        start_elem = instance.find('start')
        end_elem = instance.find('end')
        code_elem = instance.find('code')
        
        if not all([elem is not None for elem in [id_elem, start_elem, end_elem, code_elem]]):
            return None
        
        # Valores básicos
        registro = {
            'id_instancia': id_elem.text,
            'inicio_segundos': float(start_elem.text),
            'fin_segundos': float(end_elem.text),
            'codigo_completo': code_elem.text,
        }
        
        registro['duracion_segundos'] = registro['fin_segundos'] - registro['inicio_segundos']
        
        # Extraer información de labels
        jugador, equipo, periodo = extraer_labels_xml(instance)
        registro['jugador'] = jugador
        registro['equipo'] = equipo
        registro['periodo'] = periodo
        
        # Parsear código
        info_codigo = parsear_codigo_xml(code_elem.text)
        registro.update(info_codigo)
        
        return registro
        
    except Exception as e:
        return None

def extraer_labels_xml(instance) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """Extrae información de los elementos label"""
    jugador = None
    equipo = None
    periodo = None
    
    labels = instance.findall('label')
    
    for label in labels:
        text_elem = label.find('text')
        group_elem = label.find('group')
        
        if text_elem is not None:
            text_value = text_elem.text
            
            if group_elem is not None:
                group_value = group_elem.text
                
                if group_value == 'Equipo':
                    equipo = text_value
                elif group_value.startswith('Jugadores '):
                    jugador = text_value
                    if equipo is None:
                        equipo = group_value.replace('Jugadores ', '')
            else:
                if text_value in ['P1', 'P2']:
                    periodo = text_value
                elif jugador is None:
                    jugador = text_value
    
    return jugador, equipo, periodo

def parsear_codigo_xml(codigo: str) -> Dict:
    """Parsea el código XML para extraer información"""
    info = {'tipo_evento': codigo}
    
    if codigo == "Inicio":
        info['tipo_evento'] = "Inicio"
        return info
    
    # Ventanas de exigencia: "Ventana 1min y >21km/h"
    ventana_match = re.search(r'Ventana (\\d+)min y >(\\d+)km/h', codigo)
    if ventana_match:
        info['tipo_evento'] = "Ventana"
        info['duracion_ventana_min'] = int(ventana_match.group(1))
        info['velocidad_minima_kmh'] = int(ventana_match.group(2))
        return info
    
    # Alta intensidad: "Alta Intensidad (21km/h y 20m)"
    intensidad_match = re.search(r'Alta Intensidad \\((\\d+)km/h y (\\d+)m\\)', codigo)
    if intensidad_match:
        info['tipo_evento'] = "Alta Intensidad"
        info['velocidad_minima_kmh'] = int(intensidad_match.group(1))
        info['distancia_minima_m'] = int(intensidad_match.group(2))
        return info
    
    return info

# =====================================================================================
# FUNCIONES ORIGINALES MEJORADAS
# =====================================================================================

def guardar_id_en_csv(id, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    """Guarda los match_id tras leerlos del archivo ids_procesados.csv"""
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    
    # Crear directorios si no existen
    os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
    
    with open(nombre_archivo, 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([id])

def leer_ids_csv(temporada="Temporada_24_25", competicion="La_Liga", archivo_ids="ids_procesados_La_Liga.csv"):
    """Devuelve los match_id tras leerlos del archivo ids_procesados.csv"""
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    print(nombre_archivo)
    
    if not os.path.exists(nombre_archivo):
        print(f"Archivo '{nombre_archivo}' no encontrado. Creando archivo vacío.")
        os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
        with open(nombre_archivo, 'w', newline='') as file:
            pass
        return []
    else:
        with open(nombre_archivo, 'r', newline='') as file:
            reader = csv.reader(file)
            match_id_list = []
            for row in reader:
                if not row:
                    continue
                else:
                    match_id_list.append(row[0])
            return match_id_list

def obtener_ids(max_match_day, season_id, temporada, competition_id, competicion, archivo_ids):
    """Devuelve una lista con los match_ids que se jugaron antes de los max_match_day y que no están en la planilla ids_procesados existente"""
    print(temporada, competicion, archivo_ids)
    ids_existente = leer_ids_csv(temporada, competicion, archivo_ids)
    matches = ejecutar_curl_comando(f"""
    curl --location '{api_url_base}/Championships/seasons/{season_id}/competitions/{competition_id}/matches' \\
    {credenciales}
    """)

    if not matches:
        return []
    ids_filtrados = [item['id'] for item in matches if int(item['matchDayNumber']) < int(max_match_day)]
    return [id for id in ids_filtrados if id not in ids_existente]

def descargar_y_guardar_archivo_mejorado(file_url, temporada="Temporada_24_25", competicion="La_Liga", carpeta="Carpeta_General", nombre_archivo="archivo_general", extension="csv"):
    """Descarga y convierte archivos usando los procesadores robustos"""
    try:
        file_response = requests.get(file_url, timeout=30)
        
        if file_response.status_code != 200:
            print(f"❌ Error descargando: {file_response.status_code}")
            return False
        
        content = file_response.content
        
        # Crear ruta de destino (siempre CSV)
        nombre_base = nombre_archivo.rsplit('.', 1)[0] if '.' in nombre_archivo else nombre_archivo
        nombre_archivo_csv = f"{nombre_base}.csv"
        ruta_completa = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{carpeta}/{nombre_archivo_csv}'
        
        # Crear directorio
        os.makedirs(os.path.dirname(ruta_completa), exist_ok=True)
        
        # Procesar según el tipo
        if extension == 'xlsx':
            print(f"   📊 Procesando XLSX → CSV: {carpeta}/{nombre_archivo_csv}")
            return xlsx_to_csv_robusto(content, ruta_completa)
        elif extension == 'xml':
            print(f"   🔬 Procesando XML → CSV: {carpeta}/{nombre_archivo_csv}")
            return xml_to_csv_robusto(content, ruta_completa)
        elif extension == 'csv':
            print(f"   📋 Guardando CSV: {carpeta}/{nombre_archivo_csv}")
            with open(ruta_completa, 'wb') as file:
                file.write(content)
            return True
        else:
            # Fallback: guardar como está
            with open(ruta_completa, 'wb') as file:
                file.write(content)
            return True
            
    except Exception as e:
        print(f"❌ Error procesando archivo: {e}")
        return False

def descargar_archivos(ids, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    """Función principal mejorada con procesadores robustos"""
    nombres_de_archivos = []
    total_archivos = len(ids) * 8
    archivos_descargados = 0

    print(f"🚀 Iniciando descarga con procesadores robustos")
    print(f"📊 {len(ids)} partidos × 8 archivos = {total_archivos} archivos total")

    for i, id in enumerate(ids):
        print(f"\\n🏈 Procesando partido {i+1}/{len(ids)} - ID: {id}")
        
        # Obtener assets
        asset_data = ejecutar_curl_comando(f"curl --location '{api_url_base}/Assets/{id}' {credenciales}")
        if not asset_data:
            print(f"❌ No se pudieron obtener assets para {id}")
            continue

        try:
            # URLs de los 8 archivos
            file_urls = [asset_data[0]['url'], asset_data[1]['url'], asset_data[5]['url'], asset_data[9]['url'],
                         asset_data[11]['url'], asset_data[12]['url'], asset_data[13]['url'], asset_data[14]['url']]
        except (IndexError, KeyError) as e:
            print(f"❌ Error obteniendo URLs para {id}: {e}")
            continue

        # Mapeo de carpetas y extensiones
        carpetas = ['Rendimiento', 'Rendimiento', 'postpartidoequipos', 'postpartidojugador',
                    'maximaexigencia', 'maximaexigencia', 'beyondstats', 'maximaexigenciarevisada']
        extensiones = ['xlsx', 'xlsx', 'csv', 'csv', 'xlsx', 'xlsx', 'xml', 'xml']

        # Procesar cada archivo
        for j, (file_url, carpeta, extension) in enumerate(zip(file_urls, carpetas, extensiones)):
            if file_url:
                print(f"  📥 Archivo {j+1}/8: {carpeta} ({extension})")
                
                if descargar_y_guardar_archivo_mejorado(
                    file_url, 
                    temporada=temporada, 
                    competicion=competicion, 
                    carpeta=carpeta, 
                    nombre_archivo=f'archivo_{id}_descargado_{i}_{j}.{extension}',
                    extension=extension
                ):
                    nombres_de_archivos.append(f'{carpeta}/archivo_{id}_descargado_{i}_{j}.csv')
                    archivos_descargados += 1
                    porcentaje_completado = (archivos_descargados / total_archivos) * 100
                    print(f"    ✅ Progreso: {porcentaje_completado:.1f}% ({archivos_descargados}/{total_archivos})")
                else:
                    print(f"    ❌ Error procesando {carpeta}")
            else:
                print(f"  ⚠️ URL vacía para archivo {j+1}/8")
        
        # Guardar ID como procesado
        guardar_id_en_csv(id, temporada, competicion, archivo_ids)
        print(f"  ✅ Partido {id} completado")

    print(f"\\n🎉 DESCARGA COMPLETADA")
    print(f"📊 Total archivos procesados: {len(nombres_de_archivos)}")
    print(f"📁 Todos los archivos están en formato CSV")
    return nombres_de_archivos

In [7]:
# Funciones de conversión a CSV
def xlsx_to_csv(xlsx_content, csv_path):
    """Convierte contenido XLSX a CSV"""
    try:
        # Cargar el workbook desde bytes
        workbook = load_workbook(io.BytesIO(xlsx_content))
        # Usar la primera hoja
        worksheet = workbook.active
        
        # Escribir a CSV
        with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            for row in worksheet.iter_rows(values_only=True):
                # Filtrar valores None y convertir a string
                row_data = [str(cell) if cell is not None else '' for cell in row]
                writer.writerow(row_data)
        return True
    except Exception as e:
        print(f"Error al convertir XLSX a CSV: {e}")
        return False

def xml_to_csv(xml_content, csv_path):
    """Convierte contenido XML a CSV"""
    try:
        root = ET.fromstring(xml_content)
        
        # Función recursiva para extraer datos del XML
        def extract_data(element, parent_path=""):
            path = f"{parent_path}.{element.tag}" if parent_path else element.tag
            
            if element.text and element.text.strip():
                return [(path, element.text.strip())]
            
            data_items = []
            for child in element:
                data_items.extend(extract_data(child, path))
            
            return data_items
        
        # Extraer todos los datos
        all_data = extract_data(root)
        
        # Escribir a CSV
        with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Path', 'Value'])  # Headers
            writer.writerows(all_data)
        
        return True
    except Exception as e:
        print(f"Error al convertir XML a CSV: {e}")
        return False

# Función que guarda los match_id tras leerlos del archivo ids_procesados.csv
def guardar_id_en_csv(id, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    
    # Crear directorios si no existen
    os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
    
    with open(nombre_archivo, 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([id])

# Función que devuelve los match_id tras leerlos del archivo ids_procesados.csv
def leer_ids_csv(temporada="Temporada_24_25", competicion="La_Liga", archivo_ids="ids_procesados_La_Liga.csv"):
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    print(nombre_archivo)
    
    if not os.path.exists(nombre_archivo):
        print(f"Archivo '{nombre_archivo}' no encontrado. Creando archivo vacío.")
        # Crear directorios si no existen
        os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
        # Crear archivo vacío
        with open(nombre_archivo, 'w', newline='') as file:
            pass
        return []
    else:
        with open(nombre_archivo, 'r', newline='') as file:
            reader = csv.reader(file)
            match_id_list = []
            for row in reader:
                if not row:
                    continue
                else:
                    match_id_list.append(row[0])
            return match_id_list

# Función que devuelve una lista con los match_ids que se jugaron antes de los max_match_day y que no están en la planilla ids_procesados existente
def obtener_ids(max_match_day, season_id, temporada, competition_id, competicion, archivo_ids):
    print(temporada, competicion, archivo_ids)
    ids_existente = leer_ids_csv(temporada, competicion, archivo_ids)
    matches = ejecutar_curl_comando(f"""
    curl --location '{api_url_base}/Championships/seasons/{season_id}/competitions/{competition_id}/matches' \\
    {credenciales}
    """)

    if not matches:
        return []
    ids_filtrados = [item['id'] for item in matches if int(item['matchDayNumber']) < int(max_match_day)]
    return [id for id in ids_filtrados if id not in ids_existente]

# Función para descargar un archivo y convertirlo a CSV si es necesario
def descargar_y_guardar_archivo(file_url, temporada="Temporada_24_25", competicion="La_Liga", carpeta="Carpeta General", nombre_archivo="Archivo general", extension="csv"):
    file_response = requests.get(file_url)
    
    # Cambiar extensión a CSV si no lo es ya
    if not nombre_archivo.endswith('.csv'):
        nombre_base = nombre_archivo.rsplit('.', 1)[0] if '.' in nombre_archivo else nombre_archivo
        nombre_archivo_csv = f"{nombre_base}.csv"
    else:
        nombre_archivo_csv = nombre_archivo
    
    ruta_completa = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{carpeta}/{nombre_archivo_csv}'
    
    if file_response.status_code == 200:
        # Crear el directorio si no existe
        os.makedirs(os.path.dirname(ruta_completa), exist_ok=True)
        
        # Determinar el tipo de archivo y convertir si es necesario
        if extension == 'xlsx':
            # Convertir XLSX a CSV
            return xlsx_to_csv(file_response.content, ruta_completa)
        elif extension == 'xml':
            # Convertir XML a CSV
            return xml_to_csv(file_response.content, ruta_completa)
        elif extension == 'csv':
            # Ya es CSV, guardar directamente
            with open(ruta_completa, 'wb') as file:
                file.write(file_response.content)
            return True
        else:
            # Para otros formatos, intentar guardar como CSV
            with open(ruta_completa, 'wb') as file:
                file.write(file_response.content)
            return True
    return False

# Función que recibe la lista de ids a descargar y descarga los archivos de mediacoach con esa id.
# Cada Id de partido tiene 8 archivos asociados
def descargar_archivos(ids, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    nombres_de_archivos = []
    total_archivos = len(ids) * 8  # Suponiendo que cada ID tiene 8 archivos asociados.
    archivos_descargados = 0

    # Recorro lista con ids de los partidos
    for i, id in enumerate(ids):
        # Obtengo diccionario con informes de partido
        asset_data = ejecutar_curl_comando(f"curl --location '{api_url_base}/Assets/{id}' {credenciales}")
        if not asset_data:
            continue

        try:
            # Me quedo con las url que me interesan
            file_urls = [asset_data[0]['url'], asset_data[1]['url'], asset_data[5]['url'], asset_data[9]['url'],
                         asset_data[11]['url'], asset_data[12]['url'], asset_data[13]['url'], asset_data[14]['url']]
        except (IndexError, KeyError) as e:
            print(f"Error al procesar el ID {id}: {e}")
            continue  # Continúa con el siguiente ID en caso de error

        # Asigno a cada url una carpeta y una extensión de acuerdo al archivo a descargar
        carpetas = ['Rendimiento', 'Rendimiento', 'postpartidoequipos', 'postpartidojugador',
                    'maximaexigencia', 'maximaexigencia', 'beyondstats', 'maximaexigenciarevisada']
        extensiones = ['xlsx', 'xlsx', 'csv', 'csv', 'xlsx', 'xlsx', 'xml', 'xml']

        # Creo lista con diccionarios: [{file_url, carpeta, extension}] y la recorro
        for j, (file_url, carpeta, extension) in enumerate(zip(file_urls, carpetas, extensiones)):
            # Descargo el archivo y lo guardo en el directorio local en la carpeta correspondiente (todo como CSV)
            if file_url and descargar_y_guardar_archivo(
                file_url, 
                temporada=temporada, 
                competicion=competicion, 
                carpeta=carpeta, 
                nombre_archivo=f'archivo_{id}_descargado_{i}_{j}.{extension}',
                extension=extension
            ):
                nombres_de_archivos.append(f'{carpeta}/archivo_{id}_descargado_{i}_{j}.csv')
                archivos_descargados += 1
                porcentaje_completado = (archivos_descargados / total_archivos) * 100
                print(f"Progreso: {porcentaje_completado:.2f}% completado ({archivos_descargados}/{total_archivos} archivos)")
        
        # Guardo id del partido en archivo csv
        guardar_id_en_csv(id, temporada, competicion, archivo_ids)

    return nombres_de_archivos

In [8]:
# =====================================================================================
# INTERFAZ DE USUARIO CON PROCESADORES ROBUSTOS
# =====================================================================================

print("🚀 DESCARGADOR MEDIACOACH - VERSIÓN CSV LOCAL CON PROCESADORES ROBUSTOS")
print("=" * 80)
print("🔧 PROCESADORES INCLUIDOS:")
print("   📊 XLSX (Rendimiento/Máxima Exigencia): Múltiples engines + rescate zipfile")
print("   🔬 XML (Beyond Stats/Máxima Exig. Revisada): Parser específico MediaCoach")
print("   📋 CSV (Post-partido): Detección automática de delimitadores")
print("   🛡️  Resiliente a archivos corruptos y formatos problemáticos")
print("=" * 80)

# 1 - Preguntar Temporada
temporadas = [{"nombre":"Temporada 24-25", "id":"3a134240-833f-41dd-c6b0-3d6b87479c15", "input":0 },
              {"nombre":"Temporada 23-24", "id":"3a0bf8ee-7f55-aeb6-fd31-273f2d45aefa", "input":1 }]

print("\\nLista de temporadas disponibles:")
for temporada in temporadas:
    print(f'{temporada["nombre"]}, seleccione {temporada["input"]}')

input_temporadas = int(input("Seleccione una temporada. Pulse q para salir y terminar el proceso: "))
while input_temporadas not in [0,1, "q"]:
    print(f"Esa no es una opción válida, selecciona una opción entre 0 y {len(temporadas)-1}")
    input_temporadas = int(input("Seleccione una temporada: "))

if input_temporadas == 0:
    season_id = temporadas[0]["id"]
    season_name = temporadas[0]["nombre"]
elif input_temporadas == 1:
    season_id = temporadas[1]["id"]
    season_name = temporadas[1]["nombre"]

temporada = season_name.replace(" ", "_").replace("-", "_")
print(f"Ha elegido la temporada {season_name}")

# 2 - Preguntar Competición
competiciones = [{"nombre":"La Liga", "id":"39df9ec8-be91-4be5-1925-4b670a4cbed9", "input":0 },
                {"nombre":"La Liga 2", "id":"39df9ec8-becb-86ea-b5e8-600c1b47968d", "input":1 }]

print("\\nLista de competiciones disponibles:")
for competicion in competiciones:
    print(f'{competicion["nombre"]}, seleccione {competicion["input"]}')

input_competiciones = int(input("Seleccione una competición. Pulse q para salir y terminar el proceso: "))
while input_competiciones not in [0,1,"q"]:
    print(f"Esa no es una opción válida, selecciona una opción entre 0 y {len(competiciones)-1}")
    input_competiciones = int(input("Seleccione una competición: "))

if input_competiciones == 0:
    competition_id = competiciones[0]["id"]
    competition_name = competiciones[0]["nombre"]
elif input_competiciones == 1:
    competition_id = competiciones[1]["id"]
    competition_name = competiciones[1]["nombre"]

competicion = competition_name.replace(" ", "_").replace("-", "_")
print(f"Ha elegido la competición {competition_name}")

archivo_ids = f'ids_procesados_{competicion}.csv'

# 3 - Preguntar Max match days
max_match_day = input("Ingrese la cantidad de días de partidos (Ej: 1, 2, 15, etc.) o pulse q para salir y terminar el proceso: ")
while ((type(max_match_day)==int) or (max_match_day=="q")):
    print(f"Esa no es una opción válida, ingrese un número entero")
    max_match_day = int(input("Ingrese la cantidad de días de partidos (Ej: 1, 2, 15, etc.): "))
print(f"Ha elegido {max_match_day} días de partidos")

print(f"\\n{'='*80}")
print("🚀 INICIANDO DESCARGA CON PROCESADORES ROBUSTOS")
print(f"📅 Temporada: {season_name}")
print(f"🏆 Competición: {competition_name}")
print(f"📊 Días de partidos: {max_match_day}")
print(f"📁 Directorio: ./VCF_Mediacoach_Data/{temporada}/{competicion}/")
print("🔧 CARACTERÍSTICAS ROBUSTAS:")
print("   📊 XLSX → CSV: Múltiples engines (openpyxl, calamine) + rescate zipfile")
print("   🔬 XML → CSV: Parser específico para instancias MediaCoach")
print("   📋 CSV: Detección automática de delimitadores (;, ,)")
print("   🛡️  Manejo robusto de archivos corruptos")
print("   📈 Extracción de metadatos del partido (equipos, fecha, jornada)")
print(f"{'='*80}")

# 4 - Ejecutar función con datos proporcionados
ids = obtener_ids(max_match_day, season_id, temporada, competition_id, competicion, archivo_ids)
print(f"\\nSe procesarán {len(ids)} partidos")

if ids:
    print(f"🎯 IDs a procesar: {ids}")
    confirmacion = input("\\n¿Desea continuar con la descarga robusta? (s/n): ")
    if confirmacion.lower() in ['s', 'si', 'y', 'yes']:
        print("\\n🚀 Iniciando descarga con procesadores robustos...")
        print("⚠️  Nota: Los procesadores robustos manejarán automáticamente:")
        print("   • Archivos XLSX corruptos o con XML inválido")
        print("   • Múltiples engines de lectura (openpyxl, calamine)")
        print("   • Método de rescate con zipfile para casos extremos")
        print("   • Parseo específico de estructura MediaCoach")
        print("   • Extracción de metadatos de partidos")
        
        nombres_de_archivos = descargar_archivos(ids, temporada=temporada, competicion=competicion, archivo_ids=archivo_ids)
        
        print(f"\\n{'='*80}")
        print("🎉 DESCARGA ROBUSTA COMPLETADA")
        print(f"📊 Total archivos procesados: {len(nombres_de_archivos)}")
        print(f"📁 Ubicación: ./VCF_Mediacoach_Data/{temporada}/{competicion}/")
        print("✅ TODOS LOS ARCHIVOS EN FORMATO CSV:")
        print("   📊 Rendimiento: XLSX procesado → CSV con metadatos")
        print("   📈 Máxima Exigencia: XLSX procesado → CSV con metadatos")
        print("   📋 Post-partido Equipos: CSV limpio")
        print("   📋 Post-partido Jugador: CSV limpio")
        print("   🔬 Beyond Stats: XML parseado → CSV estructurado")
        print("   ⚡ Máxima Exigencia Revisada: XML parseado → CSV estructurado")
        print(f"{'='*80}")
        
        # Información adicional sobre los datos procesados
        print("\\n📋 ESTRUCTURA DE DATOS PROCESADOS:")
        print("📊 XLSX (Rendimiento/Máxima Exigencia):")
        print("   • Headers extraídos de fila 6")
        print("   • Metadatos de partido desde columna E")
        print("   • Información de equipos, fecha, jornada")
        print("🔬 XML (Beyond Stats/Máxima Exigencia Revisada):")
        print("   • Instancias con timestamps")
        print("   • Información de jugadores y equipos")
        print("   • Eventos codificados (ventanas, alta intensidad)")
        print("📋 CSV (Post-partido):")
        print("   • Datos tabulares limpios")
        print("   • Delimitadores detectados automáticamente")
        
    else:
        print("❌ Descarga cancelada por el usuario.")
else:
    print("✅ No hay partidos nuevos para procesar.")
    
print("\\n💡 PRÓXIMOS PASOS:")
print("   1. 📊 Verificar archivos CSV en las carpetas")
print("   2. 🔍 Analizar metadatos extraídos de partidos")
print("   3. 📈 Procesar datos para análisis")
print("   4. 🔄 Ejecutar nuevamente para más partidos")

🚀 DESCARGADOR MEDIACOACH - VERSIÓN CSV LOCAL CON PROCESADORES ROBUSTOS
🔧 PROCESADORES INCLUIDOS:
   📊 XLSX (Rendimiento/Máxima Exigencia): Múltiples engines + rescate zipfile
   🔬 XML (Beyond Stats/Máxima Exig. Revisada): Parser específico MediaCoach
   📋 CSV (Post-partido): Detección automática de delimitadores
   🛡️  Resiliente a archivos corruptos y formatos problemáticos
\nLista de temporadas disponibles:
Temporada 24-25, seleccione 0
Temporada 23-24, seleccione 1
Ha elegido la temporada Temporada 24-25
\nLista de competiciones disponibles:
La Liga, seleccione 0
La Liga 2, seleccione 1
Ha elegido la competición La Liga
Ha elegido 2 días de partidos
\n================================================================================
🚀 INICIANDO DESCARGA CON PROCESADORES ROBUSTOS
📅 Temporada: Temporada 24-25
🏆 Competición: La Liga
📊 Días de partidos: 2
📁 Directorio: ./VCF_Mediacoach_Data/Temporada_24_25/La_Liga/
🔧 CARACTERÍSTICAS ROBUSTAS:
   📊 XLSX → CSV: Múltiples engines (openpyxl, 

KeyboardInterrupt: 